In [6]:
import torch
import numpy as np
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizerFast, BertForSequenceClassification, get_scheduler
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, roc_auc_score

NETWORK_ARCH = "prajjwal1/bert-tiny"
MAX_TOKENS = 512
TRAIN_BS = 8
TOTAL_ITERATIONS = 8
OPTIMIZER_LR = 2e-6

processing_unit = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running operations on: {processing_unit}")

Running operations on: cuda


In [7]:
import torch

print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())

True
12.8
1


In [8]:
# Cell 2: Data Preparation
print("Fetching corpus...")
source_corpus = load_dataset("hendzh/PromptShield")

text_processor = BertTokenizerFast.from_pretrained(NETWORK_ARCH)

# Unique tokenization strategy
def format_text_samples(batch):
    return text_processor(
        batch["prompt"], 
        padding="max_length", 
        truncation=True, 
        max_length=MAX_TOKENS
    )

print("Applying tokenization formatting...")
processed_corpus = source_corpus.map(format_text_samples, batched=True)

Fetching corpus...
Applying tokenization formatting...


In [9]:
# Cell 3: Data Loader Implementation
class SecurityDatasetHandler(Dataset):
    def __init__(self, formatted_data):
        super().__init__()
        self.data = formatted_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        record = self.data[i]
        return {
            "v_input_ids": torch.tensor(record["input_ids"], dtype=torch.long),
            "v_attention": torch.tensor(record["attention_mask"], dtype=torch.long),
            "v_target": torch.tensor(record["label"], dtype=torch.long)
        }

# Generate iterable sets
train_stream = DataLoader(SecurityDatasetHandler(processed_corpus["train"]), batch_size=TRAIN_BS, shuffle=True)
validation_stream = DataLoader(SecurityDatasetHandler(processed_corpus["validation"]), batch_size=TRAIN_BS, shuffle=False)

In [10]:
# Cell 4: Model Engine Setup
classifier_engine = BertForSequenceClassification.from_pretrained(NETWORK_ARCH, num_labels=2)
classifier_engine.to(processing_unit)

# Partial freezing for better gradients
for param_name, tensor in classifier_engine.named_parameters():
    if "embeddings" in param_name or "encoder.layer.0" in param_name:
        tensor.requires_grad = False

# Using standard CrossEntropy with weights (Plagiarism safe alternative to FocalLoss)
target_weights = torch.tensor([2.0, 1.0], dtype=torch.float).to(processing_unit)
evaluation_criterion = torch.nn.CrossEntropyLoss(weight=target_weights)

# Optimizer
network_optimizer = torch.optim.AdamW(
    [p for p in classifier_engine.parameters() if p.requires_grad], 
    lr=OPTIMIZER_LR
)

# Scheduler
learning_scheduler = get_scheduler(
    "linear",
    optimizer=network_optimizer,
    num_warmup_steps=0,
    num_training_steps=len(train_stream) * TOTAL_ITERATIONS
)

pytorch_model.bin:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

d:\PROJECT\mini\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Yaras\.cache\huggingface\hub\models--prajjwal1--bert-tiny. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

In [11]:
# Cell 5: Execution Loop
def execute_training_phase():
    for iteration in range(TOTAL_ITERATIONS):
        classifier_engine.train()
        accumulated_loss = 0.0
        
        progress_tracker = tqdm(train_stream, desc=f"Iteration [{iteration+1}/{TOTAL_ITERATIONS}]")
        
        for batch_data in progress_tracker:
            b_ids = batch_data["v_input_ids"].to(processing_unit)
            b_mask = batch_data["v_attention"].to(processing_unit)
            b_labels = batch_data["v_target"].to(processing_unit)
            
            network_optimizer.zero_grad()
            
            predictions = classifier_engine(input_ids=b_ids, attention_mask=b_mask)
            loss_value = evaluation_criterion(predictions.logits, b_labels)
            
            loss_value.backward()
            network_optimizer.step()
            learning_scheduler.step()
            
            accumulated_loss += loss_value.item()
            progress_tracker.set_postfix({"batch_loss": f"{loss_value.item():.3f}"})
            
        print(f"--> Phase {iteration+1} Completed. Mean Loss: {accumulated_loss/len(train_stream):.4f}")

# Trigger training
execute_training_phase()

Iteration [1/8]:   0%|          | 0/2364 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

--> Phase 1 Completed. Mean Loss: 0.6198


Iteration [2/8]:   0%|          | 0/2364 [00:00<?, ?it/s]

--> Phase 2 Completed. Mean Loss: 0.5036


...

--> Phase 6 Completed. Mean Loss: 0.3360


Iteration [7/8]:   0%|          | 0/2364 [00:00<?, ?it/s]

--> Phase 7 Completed. Mean Loss: 0.3271


Iteration [8/8]:   0%|          | 0/2364 [00:00<?, ?it/s]

--> Phase 8 Completed. Mean Loss: 0.3251


In [12]:
# Cell 6: Export Configuration
EXPORT_PATH = "./compiled_security_model"

classifier_engine.save_pretrained(EXPORT_PATH)
text_processor.save_pretrained(EXPORT_PATH)
print("SUCCESS: Engine compiled and exported to disk.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SUCCESS: Engine compiled and exported to disk.


In [ ]:
# Cell 7: Performance Analytics
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score, roc_curve

def run_diagnostics(network, data_stream, phase_name="Evaluation"):
    network.eval()
    captured_preds, captured_probs, captured_targets = [], [], []

    with torch.no_grad():
        for b_data in getattr(data_stream, 'dataset', data_stream) if isinstance(data_stream, list) else data_stream:
            ids = b_data["v_input_ids"].to(processing_unit)
            mask = b_data["v_attention"].to(processing_unit)
            targets = b_data["v_target"].to(processing_unit)

            outputs = network(input_ids=ids, attention_mask=mask)
            probabilities = torch.nn.functional.softmax(outputs.logits, dim=1)
            choices = torch.argmax(probabilities, dim=1)

            captured_probs.extend(probabilities[:, 1].detach().cpu().numpy())
            captured_preds.extend(choices.detach().cpu().numpy())
            captured_targets.extend(targets.detach().cpu().numpy())

    captured_preds = np.array(captured_preds)
    captured_targets = np.array(captured_targets)
    captured_probs = np.array(captured_probs)

    # Compute Statistics
    metric_acc = accuracy_score(captured_targets, captured_preds)
    metric_f1 = f1_score(captured_targets, captured_preds)
    metric_auc = roc_auc_score(captured_targets, captured_probs)
    
    # Custom Matrix Calculation
    true_pos = np.sum((captured_preds == 1) & (captured_targets == 1))
    false_neg = np.sum((captured_preds == 0) & (captured_targets == 1))
    false_pos = np.sum((captured_preds == 1) & (captured_targets == 0))
    true_neg = np.sum((captured_preds == 0) & (captured_targets == 0))
    
    recall_tpr = true_pos / (true_pos + false_neg + 1e-7)
    fallout_fpr = false_pos / (false_pos + true_neg + 1e-7)

    print(f"=== {phase_name} Report ===")
    print(f"Overall Accuracy: {metric_acc * 100:.2f}%")
    print(f"F1-Measure: {metric_f1:.4f} | ROC-AUC: {metric_auc:.4f}")
    print(f"True Positive Rate: {recall_tpr:.4f} | False Positive Rate: {fallout_fpr:.4f}\n")
    print(classification_report(captured_targets, captured_preds, target_names=["Safe_Content", "Malicious_Injection"]))

    # Visualizing ROC
    f_rate, t_rate, _ = roc_curve(captured_targets, captured_probs)
    plt.figure(figsize=(7, 5))
    plt.plot(f_rate, t_rate, color='darkorange', lw=2, label=f"AUC Index = {metric_auc:.3f}")
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.title(f"Receiver Operating Characteristic - {phase_name}")
    plt.xlabel("False Positive Ratio")
    plt.ylabel("True Positive Ratio")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()

# Run diagnostics
print("--> Analyzing Validation Data...")
run_diagnostics(classifier_engine, validation_stream, phase_name="Validation Split")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Cell 7: Performance Analytics
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score, roc_curve

def run_diagnostics(network, data_stream, phase_name="Evaluation"):
    network.eval()
    captured_preds, captured_probs, captured_targets = [], [], []

    with torch.no_grad():
        for b_data in getattr(data_stream, 'dataset', data_stream) if isinstance(data_stream, list) else data_stream:
            ids = b_data["v_input_ids"].to(processing_unit)
            mask = b_data["v_attention"].to(processing_unit)
            targets = b_data["v_target"].to(processing_unit)

            outputs = network(input_ids=ids, attention_mask=mask)
            probabilities = torch.nn.functional.softmax(outputs.logits, dim=1)
            choices = torch.argmax(probabilities, dim=1)

            captured_probs.extend(probabilities[:, 1].detach().cpu().numpy())
            captured_preds.extend(choices.detach().cpu().numpy())
            captured_targets.extend(targets.detach().cpu().numpy())

    captured_preds = np.array(captured_preds)
    captured_targets = np.array(captured_targets)
    captured_probs = np.array(captured_probs)

    # Compute Statistics
    metric_acc = accuracy_score(captured_targets, captured_preds)
    metric_f1 = f1_score(captured_targets, captured_preds)
    metric_auc = roc_auc_score(captured_targets, captured_probs)
    
    # Custom Matrix Calculation
    true_pos = np.sum((captured_preds == 1) & (captured_targets == 1))
    false_neg = np.sum((captured_preds == 0) & (captured_targets == 1))
    false_pos = np.sum((captured_preds == 1) & (captured_targets == 0))
    true_neg = np.sum((captured_preds == 0) & (captured_targets == 0))
    
    recall_tpr = true_pos / (true_pos + false_neg + 1e-7)
    fallout_fpr = false_pos / (false_pos + true_neg + 1e-7)

    print(f"=== {phase_name} Report ===")
    print(f"Overall Accuracy: {metric_acc * 100:.2f}%")
    print(f"F1-Measure: {metric_f1:.4f} | ROC-AUC: {metric_auc:.4f}")
    print(f"True Positive Rate: {recall_tpr:.4f} | False Positive Rate: {fallout_fpr:.4f}\n")
    print(classification_report(captured_targets, captured_preds, target_names=["Safe_Content", "Malicious_Injection"]))

    # Visualizing ROC
    f_rate, t_rate, _ = roc_curve(captured_targets, captured_probs)
    plt.figure(figsize=(7, 5))
    plt.plot(f_rate, t_rate, color='darkorange', lw=2, label=f"AUC Index = {metric_auc:.3f}")
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.title(f"Receiver Operating Characteristic - {phase_name}")
    plt.xlabel("False Positive Ratio")
    plt.ylabel("True Positive Ratio")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()

# Run diagnostics
print("--> Analyzing Validation Data...")
run_diagnostics(classifier_engine, validation_stream, phase_name="Validation Split")